In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Analysis Settings
P_ENTER = 0.05          # p-value threshold to ADD a variable
P_REMOVE = 0.10         # p-value threshold to REMOVE a variable
VIF_THRESHOLD = 2.0     # Maximum VIF allowed

print(f"Settings: p_enter<{P_ENTER} | p_remove>{P_REMOVE} | VIF<{VIF_THRESHOLD}")

In [ ]:
# File Paths
loadings_file = "C:/Users/ahusse12/OneDrive - Kent State University/Documents/Stepwise-Learning/data/test/test_loadings.csv"
library_file  = "C:/Users/ahusse12/OneDrive - Kent State University/Documents/Stepwise-Learning/data/test/test_library.csv"
output_dir    = "C:/Users/ahusse12/OneDrive - Kent State University/Documents/Stepwise-Learning/output"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

print("Loadings:", loadings_file)
print("Library :", library_file)
print("Output  :", output_dir + "/")

In [ ]:
# Load VPCA loadings
loadings = pd.read_csv(loadings_file)
loadings = loadings.set_index('Wavelength')
loadings = loadings.drop(['Eigenvalues', 'Frac Var', 'Extracted Frac Var'], errors='ignore')
vpc_cols = [col for col in loadings.columns if col.startswith('VPC')]
loadings = loadings[vpc_cols]
loadings = loadings.apply(pd.to_numeric, errors='coerce')
loadings.index = pd.to_numeric(loadings.index, errors='coerce')
loadings = loadings[loadings.index.notna()]

print("VPCA Loadings:")
print(loadings)
print("\nShape: {} wavelengths × {} VPCs".format(loadings.shape[0], loadings.shape[1]))

In [ ]:
# Load spectral library (robust to professor's format)
library = pd.read_csv(library_file)

# Auto-detect wavelength index column (handles 'Wavelength' or '?_centre_nm')
wav_col = next((c for c in library.columns
                if 'wavelength' in c.lower() or c.startswith('?')),
               library.columns[0])
library = library.set_index(wav_col)
library.index = pd.to_numeric(library.index, errors='coerce')
library = library[library.index.notna()]

# Convert to numeric, drop fully empty columns
library = library.apply(pd.to_numeric, errors='coerce')
library = library.dropna(axis=1, how='all')

# Strip Z-score suffixes from column names → clean names
library.columns = [c.replace('_Zdrdl', '').replace(' Zdrdl', '').strip()
                   for c in library.columns]

# Drop metadata columns (VPC loadings, Band, duplicate wavelength cols starting with ?)
drop_cols = [c for c in library.columns
             if 'VPC' in c or c.startswith('IRL') or c == 'Band' or c.startswith('?')]
library = library.drop(columns=drop_cols, errors='ignore')

# Align wavelengths with loadings
loadings.index = pd.Index(np.round(loadings.index.values, 1))
library.index  = pd.Index(np.round(library.index.values, 1))
common_wl = sorted(set(loadings.index) & set(library.index))
loadings = loadings.loc[common_wl]
library  = library.loc[common_wl]

print("Spectral Library:")
print(library)
print("\nShape: {} wavelengths × {} materials".format(library.shape[0], library.shape[1]))
print("Common wavelengths:", common_wl)

In [ ]:
# Z-score standardization
def zscore_dataframe(df):
    """Convert each column to z-scores (mean=0, std=1) using population std."""
    cols = {}
    for col in df.columns:
        vals = df[col].values.astype(float)
        m, s = np.mean(vals), np.std(vals, ddof=0)
        if s == 0:
            cols[col] = np.zeros(len(vals))
        else:
            cols[col] = (vals - m) / s
    return pd.DataFrame(cols, index=df.index)

z_loadings = zscore_dataframe(loadings)
z_library  = zscore_dataframe(library)

# Rename VPC columns to VPCA_Z1, VPCA_Z2, etc.
z_loadings.columns = [f'VPCA_Z{i+1}' for i in range(len(z_loadings.columns))]

print("="*70)
print("Z-SCORED VPCA LOADINGS")
print("="*70)
print(z_loadings.round(4))
print(f"\nShape: {z_loadings.shape[0]} wavelengths × {z_loadings.shape[1]} components")

print("\n" + "="*70)
print("Z-SCORED SPECTRAL LIBRARY")
print("="*70)
print(z_library.round(4))
print(f"\nShape: {z_library.shape[0]} wavelengths × {z_library.shape[1]} materials")

# Descriptive Statistics Table
all_z = pd.concat([z_loadings, z_library], axis=1)
desc_rows = []
for col in all_z.columns:
    m = all_z[col].mean()
    s = all_z[col].std(ddof=0)
    desc_rows.append({
        'Variable': col,
        'N': len(all_z),
        'Mean': round(m, 6),
        'Std Dev': round(s, 6),
        'Check': '✓' if abs(m) < 1e-9 and abs(s - 1.0) < 1e-9 else '⚠'
    })
desc_df = pd.DataFrame(desc_rows)

print("\n" + "="*70)
print("DESCRIPTIVE STATISTICS (Z-SCORED VARIABLES)")
print("="*70)
print("All variables should have Mean ≈ 0 and Std Dev = 1")
print("-"*70)
print(desc_df.to_string(index=False))

In [ ]:
# VIF Calculation Function
def calculate_vif(X_subset):
    """Calculate Variance Inflation Factor for each variable."""
    Xc = sm.add_constant(X_subset)
    return pd.DataFrame([
        {'Variable': col, 'VIF': round(variance_inflation_factor(Xc.values, i), 3)}
        for i, col in enumerate(Xc.columns) if col != 'const'
    ])

print("✓ calculate_vif() defined")

In [ ]:
# Bidirectional Stepwise Regression with History Tracking
def stepwise_bidirectional(y, X, p_enter=0.05, p_remove=0.10,
                           vif_threshold=100.0, verbose=True):
    """
    Bidirectional stepwise regression with VIF control.
    Returns: (selected_vars, history)
    """
    selected  = []
    remaining = list(X.columns)
    history   = []
    step      = 0

    if verbose:
        print(f"Bidirectional Stepwise | p_enter<{p_enter} | p_remove>{p_remove} | VIF<{vif_threshold}")

    for _ in range(100):
        made_change = False

        # FORWARD STEP: Find best variable to add
        best_p, best_var, best_r2 = 1.0, None, 0.0
        for var in remaining:
            try:
                m = sm.OLS(y, sm.add_constant(X[selected + [var]])).fit()
                if m.pvalues[var] < best_p:
                    best_p, best_var, best_r2 = m.pvalues[var], var, m.rsquared
            except Exception:
                continue

        if best_var and best_p < p_enter:
            test = selected + [best_var]
            max_vif = calculate_vif(X[test])['VIF'].max() if len(test) > 1 else 1.0
            
            if max_vif <= vif_threshold:
                step += 1
                selected.append(best_var)
                remaining.remove(best_var)
                made_change = True
                history.append({
                    'Step': step,
                    'Action': 'ADD',
                    'Variable': best_var,
                    'P_value': round(best_p, 4),
                    'R_squared': round(best_r2, 4),
                    'VIF': round(max_vif, 2)
                })
                if verbose:
                    print(f"Step {step}: + ADD    {best_var:<25} | p={best_p:.4f} | R²={best_r2:.4f} | VIF={max_vif:.2f}")
            else:
                remaining.remove(best_var)
                if verbose:
                    print(f"  SKIP  {best_var:<25} | p={best_p:.4f} but VIF={max_vif:.2f} > {vif_threshold}")

        # BACKWARD STEP: Remove worst variable by p-value
        if selected:
            try:
                m = sm.OLS(y, sm.add_constant(X[selected])).fit()
                pvals = m.pvalues.drop('const', errors='ignore')
                worst_var, worst_p = pvals.idxmax(), pvals.max()
                
                if worst_p > p_remove:
                    step += 1
                    selected.remove(worst_var)
                    remaining.append(worst_var)
                    made_change = True
                    history.append({
                        'Step': step,
                        'Action': 'REMOVE',
                        'Variable': worst_var,
                        'P_value': round(worst_p, 4),
                        'R_squared': None,
                        'VIF': None
                    })
                    if verbose:
                        print(f"Step {step}: - REMOVE {worst_var:<25} | p={worst_p:.4f} > {p_remove}")
            except Exception:
                pass

            # BACKWARD STEP: Remove highest VIF variable
            if len(selected) > 1:
                vif_df = calculate_vif(X[selected])
                worst = vif_df.loc[vif_df['VIF'].idxmax()]
                
                if worst['VIF'] > vif_threshold:
                    step += 1
                    selected.remove(worst['Variable'])
                    remaining.append(worst['Variable'])
                    made_change = True
                    history.append({
                        'Step': step,
                        'Action': 'REMOVE_VIF',
                        'Variable': worst['Variable'],
                        'P_value': None,
                        'R_squared': None,
                        'VIF': round(worst['VIF'], 2)
                    })
                    if verbose:
                        print(f"Step {step}: - REMOVE {worst['Variable']:<25} | VIF={worst['VIF']:.2f} > {vif_threshold}")

        if not made_change:
            if verbose:
                print("\nConverged.")
            break

    # Final summary
    if verbose:
        if selected:
            fm = sm.OLS(y, sm.add_constant(X[selected])).fit()
            vdf = calculate_vif(X[selected]) if len(selected) > 1 else pd.DataFrame([{'Variable': selected[0], 'VIF': 1.0}])
            print(f"\nFinal Model: {len(selected)} variable(s) | R²={fm.rsquared:.4f}")
            for _, row in vdf.iterrows():
                print(f"  {row['Variable']:<25} VIF={row['VIF']:.2f}")
        else:
            print("\nNo variables selected")

    return selected, history

print("✓ stepwise_bidirectional() defined")

In [ ]:
print("\n" + "="*70)

all_results = []

for vpc in z_loadings.columns:
    print(f"\n{'='*70}\nAnalyzing {vpc}\n{'='*70}")
    
    selected, history = stepwise_bidirectional(
        y=z_loadings[vpc],
        X=z_library,
        p_enter=P_ENTER,
        p_remove=P_REMOVE,
        vif_threshold=VIF_THRESHOLD,
        verbose=True
    )
    
    if selected:
        model = sm.OLS(z_loadings[vpc], sm.add_constant(z_library[selected])).fit()
        r2, adj_r2 = model.rsquared, model.rsquared_adj
        
        if len(selected) > 1:
            vif_df = calculate_vif(z_library[selected])
            max_vif = vif_df['VIF'].max()
            vif_details = ', '.join([f"{r['Variable']}({r['VIF']:.2f})" for _, r in vif_df.iterrows()])
        else:
            max_vif = 1.0
            vif_details = f"{selected[0]}(1.00)"
    else:
        r2 = adj_r2 = max_vif = 0.0
        vif_details = 'N/A'
    
    all_results.append({
        'Component': vpc,
        'N_Materials': len(selected),
        'Materials': ', '.join(selected) if selected else 'None',
        'R_squared': round(r2, 4),
        'Adj_R_squared': round(adj_r2, 4),
        'Max_VIF': round(max_vif, 2),
        'VIF_Details': vif_details
    })

results_df = pd.DataFrame(all_results)
print("\n" + "="*70)
print(results_df.to_string(index=False))
print("="*70)

In [ ]:
# ================================================================
# BUILD REGRESSION TABLES (INCREMENTAL MODELS)
# ================================================================

from scipy.stats import f as f_dist

def build_regression_tables(vpc_name, y, X, selected, history):
    """
    Build Model Summary, ANOVA, and Coefficients tables.
    Each model = one more variable added from the final selected list.
    Returns: (model_summary_df, anova_df, coefficients_df)
    """
    n = len(y)

    # Zero-order correlation: Pearson r of each predictor with y
    zero_order = {col: round(float(y.corr(X[col])), 4) for col in X.columns}

    # Determine add-order of the final selected variables from history
    add_order = [h['Variable'] for h in history if h['Action'] == 'ADD']
    seen = set()
    sel_ordered = [v for v in add_order
                   if v in selected and v not in seen and not seen.add(v)]

    summary_rows, anova_rows, coef_rows = [], [], []
    prev_r2 = 0.0

    for i in range(len(sel_ordered)):
        vars_in = sel_ordered[:i + 1]
        k       = len(vars_in)
        model   = sm.OLS(y, sm.add_constant(X[vars_in])).fit()

        r2    = model.rsquared
        r2chg = r2 - prev_r2
        df1, df2 = 1, n - k - 1
        f_chg  = (r2chg / df1) / ((1 - r2) / df2) if df2 > 0 and (1 - r2) > 0 else None
        sig_f  = round(float(f_dist.sf(f_chg, df1, df2)), 4) if f_chg else None

        # Model Summary
        summary_rows.append({
            'Dependent':    vpc_name,
            'Model':        i + 1,
            'R':            round(float(np.sqrt(r2)), 4),
            'R Square':     round(r2, 4),
            'Adj R Square': round(model.rsquared_adj, 4),
            'Std Error':    round(float(np.sqrt(model.mse_resid)), 4),
            'R Sq Change':  round(r2chg, 4),
            'F Change':     round(f_chg, 3) if f_chg else None,
            'df1':          df1,
            'df2':          df2,
            'Sig F Change': sig_f,
        })

        # ANOVA
        ms_reg = model.ess / k
        ms_res = model.ssr / (n - k - 1)
        anova_rows += [
            {'Model': i+1, 'Dependent': vpc_name, 'Source': 'Regression',
             'Sum of Squares': round(model.ess, 4), 'df': k,
             'Mean Square': round(ms_reg, 4),
             'F': round(model.fvalue, 3), 'Sig': round(model.f_pvalue, 4)},
            {'Model': i+1, 'Dependent': vpc_name, 'Source': 'Residual',
             'Sum of Squares': round(model.ssr, 4), 'df': n - k - 1,
             'Mean Square': round(ms_res, 4), 'F': None, 'Sig': None},
            {'Model': i+1, 'Dependent': vpc_name, 'Source': 'Total',
             'Sum of Squares': round(model.centered_tss, 4), 'df': n - 1,
             'Mean Square': None, 'F': None, 'Sig': None},
        ]

        # Coefficients (standardized coefficients only, no constant)
        vif_map = ({r['Variable']: r['VIF']
                    for _, r in calculate_vif(X[vars_in]).iterrows()}
                   if k > 1 else {vars_in[0]: 1.0})
        for var in vars_in:
            coef_rows.append({
                'Dependent':    vpc_name,
                'Model':        i + 1,
                'Variable':     var,
                'Std Beta':     round(float(model.params[var]), 4),
                'Std Error':    round(float(model.bse[var]), 4),
                't':            round(float(model.tvalues[var]), 3),
                'Sig':          round(float(model.pvalues[var]), 4),
                'Zero-order r': zero_order.get(var),
                'VIF':          round(vif_map.get(var, 1.0), 3),
            })

        prev_r2 = r2

    return (pd.DataFrame(summary_rows),
            pd.DataFrame(anova_rows),
            pd.DataFrame(coef_rows))


# ================================================================
# GENERATE AND DISPLAY TABLES FOR ALL VPCs
# ================================================================

print("\n" + "="*90)
print("REGRESSION ANALYSIS TABLES (INCREMENTAL MODELS)")
print("="*90)

excel_file = os.path.join(output_dir, 'regression_analysis_results.xlsx')
vpc_tables = {}

for result in all_results:
    vpc = result['Component']
    
    if result['N_Materials'] == 0:
        print(f"\n{vpc}: No variables selected")
        continue
    
    selected = [m.strip() for m in result['Materials'].split(',')]
    
    # Re-run stepwise to get history
    y = z_loadings[vpc]
    _, history = stepwise_bidirectional(
        y=y, X=z_library,
        p_enter=P_ENTER, p_remove=P_REMOVE,
        vif_threshold=VIF_THRESHOLD, verbose=False
    )
    
    summary_df, anova_df, coef_df = build_regression_tables(
        vpc_name=vpc,
        y=y,
        X=z_library,
        selected=selected,
        history=history
    )
    
    vpc_tables[vpc] = (summary_df, anova_df, coef_df)
    
    print(f"\n{'='*90}")
    print(f"{vpc}")
    print("="*90)
    
    print("\nModel Summary:")
    print(summary_df.to_string(index=False))
    
    print("\n\nANOVA:")
    print(anova_df.to_string(index=False))
    
    print("\n\nCoefficients:")
    print(coef_df.to_string(index=False))
    print()

# Save to Excel with multiple sheets
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    for vpc, (ms_df, anova_df, coef_df) in vpc_tables.items():
        row = 0
        ms_df.to_excel(writer, sheet_name=vpc, startrow=row, index=False)
        row += len(ms_df) + 3
        anova_df.to_excel(writer, sheet_name=vpc, startrow=row, index=False)
        row += len(anova_df) + 3
        coef_df.to_excel(writer, sheet_name=vpc, startrow=row, index=False)
    
    # Summary sheet
    results_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Descriptive stats
    desc_df.to_excel(writer, sheet_name='Descriptive_Stats', index=False)

print("\n" + "="*90)
print(f"✓ All tables saved to: {excel_file}")
print("="*90)